In [1]:
import numpy as np
import tensorflow as tf
from keras import Sequential
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, LSTM, Dropout, Bidirectional
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences


In [2]:
# 读取周杰伦歌词数据
with open('jaychou_lyrics.txt', 'r', encoding='utf-8') as f:
    lyrics = f.readlines()

In [3]:
#文本预处理
tokenizer = Tokenizer(char_level=True) #char_level=True表示按字分词，初始化Tokenizer文本预处理对象
tokenizer.fit_on_texts(lyrics) # 根据输入的文本生成词典，将每个词映射为整数索引
print(tokenizer.word_index) # 打印出词表

total_words = len(tokenizer.word_index) + 1 # 总字数
print(total_words)

# 将每句歌词的词转为索引值，并生成输入序列
input_sequences = [] # 输入序列
for line in lyrics:
    token_list = tokenizer.texts_to_sequences([line])[0] # 将每一句可词转为一个整数序列
    print(token_list)
    for i in range(1, len(token_list)): # 遍历每句歌词
        n_gram_sequence = token_list[:i] # 去除前i个词
        # 生成所有可能的n-gram序列，如果歌词为“我爱你”，那么会生成“我”， “我爱”， “我爱你”
        input_sequences.append(n_gram_sequence)
#print(input_sequences)

# 填充序列，找到最长的序列长度，然后填充到所有序列到同一长度
max_sequence_len = max([len(x) for x in input_sequences])
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre'))
print(input_sequences)
# 创建输入和输出数据
X = input_sequences[:, :-1] #将每个n-gram序列的前n-1个词作为输入数据
y = input_sequences[:, -1] # 将每个n-gram序列的最后一个词作为输出数据
# 将y转换为one-hot编码 categorical_crossentropy损失函数需要one-hot编码
y = tf.keras.utils.to_categorical(y, num_classes=total_words)

{'\n': 1, ' ': 2, '的': 3, '我': 4, '你': 5, '不': 6, '一': 7, '在': 8, '了': 9, '是': 10, '有': 11, '\u3000': 12, '着': 13, '人': 14, '想': 15, '过': 16, '爱': 17, '这': 18, '说': 19, '要': 20, '开': 21, '那': 22, '就': 23, '到': 24, '会': 25, '来': 26, '没': 27, '天': 28, '上': 29, '么': 30, '回': 31, '能': 32, '好': 33, '手': 34, '都': 35, '看': 36, '只': 37, '风': 38, '得': 39, '用': 40, '让': 41, '们': 42, '去': 43, '心': 44, '下': 45, '再': 46, '还': 47, '地': 48, '为': 49, '里': 50, '对': 51, '道': 52, '如': 53, '个': 54, '时': 55, '后': 56, '也': 57, '笑': 58, '等': 59, '多': 60, '无': 61, '很': 62, '走': 63, '可': 64, '离': 65, '像': 66, '太': 67, '情': 68, '起': 69, '中': 70, '出': 71, '被': 72, '他': 73, '谁': 74, '远': 75, '眼': 76, '面': 77, '小': 78, '生': 79, '已': 80, '自': 81, '听': 82, '空': 83, '快': 84, '却': 85, '雨': 86, '什': 87, '记': 88, '美': 89, '泪': 90, '子': 91, '别': 92, '跟': 93, '色': 94, '感': 95, '知': 96, '光': 97, '事': 98, '气': 99, '样': 100, '大': 101, '身': 102, '她': 103, '口': 104, '放': 105, '以': 106, '前': 107, '见': 108, '给': 109, '最': 110, '

In [11]:
# 构建模型
model = Sequential() # 创建一个顺序模型
# 词嵌入层， 将每个词转为100纬的词向量
model.add(Embedding(total_words, 100, input_length=max_sequence_len-1))
# 添加第一层双向LSTM， 隐藏状态和细胞状态是一个256维向量，return_sequences = True表示该层会返回隐藏状态，以供后续处理
model.add(Bidirectional(LSTM(256, return_sequences=True)))
# Dropout层，防止过拟合，随机丢弃20%的隐藏状态
model.add(Dropout(0.2))
# 第二层双向LSTM，隐藏状态和细胞状态是256维向量，只返回输出，不返回隐藏状态
model.add(Bidirectional(LSTM(256)))

# 全链接层，输出唯独词汇表的大小，softmax激活函数用于处理多酚类问题，用于返回每个概率分布
model.add(Dense(total_words, activation='softmax'))

In [12]:
# 编译模型
# 损失函数; categorical_crossentropy: 多分类任务常用的损失函数
# 优化器： adam： 一种优化器，常用于训练深度学习模型
# 评估指标： accuracy: 用于衡量训练过程中的准备率
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# 训练模型
# X是训练数据， y是目前数据， epochs是训练的轮数， batch_size是每次训练的样本数， verbose 显示训练进度
model.fit(X, y, epochs=10, batch_size=64, verbose=1)


Epoch 1/10
898/898 ━━━━━━━━━━━━━━━━━━━━ 94s 104ms/step - accuracy: 0.0548 - loss: 6.5315
Epoch 2/10
898/898 ━━━━━━━━━━━━━━━━━━━━ 103s 115ms/step - accuracy: 0.0764 - loss: 5.9781
Epoch 3/10
898/898 ━━━━━━━━━━━━━━━━━━━━ 104s 116ms/step - accuracy: 0.0903 - loss: 5.6169
Epoch 4/10
898/898 ━━━━━━━━━━━━━━━━━━━━ 97s 108ms/step - accuracy: 0.1209 - loss: 5.2064
Epoch 5/10
898/898 ━━━━━━━━━━━━━━━━━━━━ 100s 111ms/step - accuracy: 0.1478 - loss: 4.8380
Epoch 6/10
898/898 ━━━━━━━━━━━━━━━━━━━━ 101s 112ms/step - accuracy: 0.1892 - loss: 4.4554
Epoch 7/10
898/898 ━━━━━━━━━━━━━━━━━━━━ 100s 112ms/step - accuracy: 0.2306 - loss: 4.1347
Epoch 8/10
898/898 ━━━━━━━━━━━━━━━━━━━━ 100s 111ms/step - accuracy: 0.2747 - loss: 3.8423
Epoch 9/10
898/898 ━━━━━━━━━━━━━━━━━━━━ 99s 111ms/step - accuracy: 0.3141 - loss: 3.5899
Epoch 10/10
898/898 ━━━━━━━━━━━━━━━━━━━━ 96s 107ms/step - accuracy: 0.3539 - loss: 3.3611


In [13]:
# 生成歌词
seed_text = "我爱" # 种子文本
next_words = 100 # 生成多少个字
temperature = 1 # 温度参数， 控制生成的文本的多样性，值越小，生成的文本越随机，值越大，生成的文本越保守

# 每次循环生成一个词，直到生成next_words个词
for _ in range(next_words):
    # 将种子文本转换为整数序列
    token_list = tokenizer.texts_to_sequences([seed_text])[0]
    # 将输入的token_list填充到指定长度
    token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')

    # 使用训练好的模型预测下一个词的概率分布, model_predict()返回一个二维数组，每个元素表示一个词的概率
    # 我们只预测单个输出， 所以取二维数组的第一个元素
    predictions = model.predict(token_list, verbose=0)[0]

    #  调整预测的概率分布，转为np数组，方便计算
    predictions = np.asarray(predictions).astype('float64')
    # 对概率分布进行对数运算，加上一个小常熟，避免计算出负无穷，对数变换有助于增强差异，特别是处理极小概率
    # 当温度等于1，概率分布保持原样
    # 当温度大于1，概率分布变得更加平坦，生成的文本更随机
    # 当温度小于1， 概率分布变得更加尖锐， 生成的文本更保守
    predictions = np.log(predictions + 1e-7) / temperature

    # 对经过对数处理的概率重新取指数，得到一个归一化的概率分布
    predictions = np.exp(predictions) / np.sum(np.exp(predictions))

    # 随机选择一个词作为预测结果
    predicted = np.random.choice(range(total_words), p = predictions)

    output_word = ""

    for word, index in tokenizer.word_index.items():
        if index == predicted:
            output_word = word
            break
    seed_text += output_word

print(seed_text)

我爱店重重写了面方　飞开为正　刚雨不欣赏到深时对太隐类灯在远距牛药 白截不对见了 才舍的你更拉会看 最梯远不像勇诉我看办不见活心萎的痕不看得到走不得我离多才该不穿受听你才能回要我不断过我要随 咆寂酷了多疲
